In [1]:
import json
import os
import time
from google import genai
from google.genai import types
import datetime
import asyncio


with open('keys.json', 'r') as file:
    keys = json.load(file)

### Grounding the search for specific hardware EOS. 

In [2]:
with open('prompt.txt', 'r') as pmt_file:
    instruct = pmt_file.read()

with open('promptTest.txt', 'r') as pmt_file:
    instruct_test = pmt_file.read()

In [3]:
def client_setup():
    client = genai.Client(api_key=keys['GEMINI_API_KEY'])

    # Set up the google search tool for the client.
    scraper_client = types.Tool(google_search=types.GoogleSearch())
    config = types.GenerateContentConfig(
        tools=[scraper_client],
        #thinking_config=types.ThinkingConfig(thinking_budget=-1), # swtich off if non-thinking model is used
        temperature = 0,
        top_p=1,
)
    return client, config

In [ ]:
# prompt prototype do not run

client, config = client_setup()
response = client.models.generate_content(
    model="gemini-2.0-flash-001",
    contents='What is Nutanix. Write a poem.',
    config=config
)
print(response.text)

Nutanix is a cloud computing company that specializes in software for data centers and hybrid multi-cloud deployments. It is a leader in hyper-converged infrastructure (HCI) and enterprise cloud solutions. Nutanix combines computing, storage, and virtualization into an integrated system that can be managed from a single platform. Their software-defined architecture adapts to various hardware and cloud options, offering flexibility, scalability, and resilience.

Here's a poem about Nutanix:

In clouds of data, a need takes hold,
For systems simple, stories to be told.
Nutanix rises, a platform so grand,
Across hybrid landscapes, a helping hand.

It merges the pieces, compute, storage, and more,
Virtualization's magic, right to the core.
One platform to manage, with ease and with grace,
In the digital realm, it finds its place.

From private to public, the edge it does roam,
Adapting and scaling, like a digital gnome.
So hail Nutanix, in the cloud's vast domain,
Simplifying IT, again and

: 

### Processing the HW File

In [5]:
import pandas as pd
import time
from classes import Helper, Cleaner, Processing

In [6]:
# read the file in 
from classes import Helper

df = Helper.preprocess('SWandHW.xlsx', sheet='Sheet1')

Starting Preprocessing...
Asset list read in 31.3s
-----------------------------------


In [7]:
omnii_sw = Cleaner.clean_text_to_unique(df["Software Version"])
omnii_hw = Cleaner.clean_text_to_unique(df["Hardware Brand/Model"])

omnii_sw

array(['ekm 1.10', 'mobimed 4.8.1', '7zip 19.00',
       'google chrome 115.0.5790.102', 'splunk universal fowarder 9.0.2',
       'windows server 2016', 'sql server 2016', 'ekm agent 8.4.2',
       'sql server management studio v19.1', 'jboss eap 7.3',
       'java 8 jre v371', 'ibm websphere mq explorer 9.3.2',
       'omnii gateway v1.1.0', 'display terminal 1.2.1',
       'windows server 2019', 'audit app 1.1.1',
       'solarwinds sem agent 2023.2.1', 'powerbi 2.93.641.0',
       'omnii gateway v1.0.0', 'sem 2023.2.1', 'ibm isam aac 10.0.2',
       'ibm isam rp 10.0.2', 'telemedicine 1.0.0'], dtype=object)

In [8]:
omnii_sw

array(['ekm 1.10', 'mobimed 4.8.1', '7zip 19.00',
       'google chrome 115.0.5790.102', 'splunk universal fowarder 9.0.2',
       'windows server 2016', 'sql server 2016', 'ekm agent 8.4.2',
       'sql server management studio v19.1', 'jboss eap 7.3',
       'java 8 jre v371', 'ibm websphere mq explorer 9.3.2',
       'omnii gateway v1.1.0', 'display terminal 1.2.1',
       'windows server 2019', 'audit app 1.1.1',
       'solarwinds sem agent 2023.2.1', 'powerbi 2.93.641.0',
       'omnii gateway v1.0.0', 'sem 2023.2.1', 'ibm isam aac 10.0.2',
       'ibm isam rp 10.0.2', 'telemedicine 1.0.0'], dtype=object)

In [9]:
import numpy as np
test_list = omnii_sw[17:]
test_list = np.append(test_list, ['Pan-OS 11.1'])
test_list

array(['powerbi 2.93.641.0', 'omnii gateway v1.0.0', 'sem 2023.2.1',
       'ibm isam aac 10.0.2', 'ibm isam rp 10.0.2', 'telemedicine 1.0.0',
       'Pan-OS 11.1'], dtype=object)

### Gemini API and JSON parsing

In [10]:
import asyncio
from classes import Helper, Cleaner, Processing

def error_cache(results, eos_list):
    """
    Caches the results and eos_list for items that failed to process.
    """
    success, unsuccess = [], []
    for i, result in enumerate(results):
        if result is not None:
            success.append(result)
        else:
            unsuccess.append(eos_list[i])  # This mapping is always correct
    return success, unsuccess

async def process_line(string, client, config):
    print(f"Processing item: {string}")
    try:
        await asyncio.sleep(2)  # Sleep to avoid hitting rate limits
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            #contents=instruct_test + ' ' + string,
            contents=instruct + ' ' + string,
            config=config
        )
        return Helper.parse_llm_json(response.text) # includes throwing none in here as well
    except Exception as e:
        print(f"Error processing {string}: {e}")
        return None # need to add some handling here to log if needed. 

async def main(eos_list):
    client, config = client_setup()
    tasks = [process_line(item, client, config) for item in eos_list]
    results = await asyncio.gather(*tasks)

    # error caching for failed API calls or bad responses
    success, unsuccess = error_cache(results, eos_list)

    print(f"Successfully processed {len(success)} items.")
    
    # Add a retry limit if needed
    retry_limit = 3
    retry_count = 0

    while unsuccess and retry_count < retry_limit:
        print(f"Retrying {len(unsuccess)} items...")
        retry_tasks = [process_line(item, client, config) for item in unsuccess]
        retry_results = await asyncio.gather(*retry_tasks)
        retry_success, unsuccess = error_cache(retry_results, unsuccess)
        
        # add to main success list
        success.extend(retry_success)

        retry_count += 1
        if unsuccess:
            print(f"Retry {retry_count} failed for {len(unsuccess)} items.")
    
    return success, unsuccess

async def run_async(lst):
    print("Starting async processing...")
    # add time start
    start_time = time.time()
    results, failed_items = await main(lst)
    # add time end
    elapsed = time.time() - start_time  # Calculate elapsed time
    print(f"Time taken: {elapsed:.2f} seconds")  # Print elapsed time
    print("Async processing completed.")
    return results, failed_items 

# Run the async function, change this for the script. 
results, failed_items = await run_async(test_list)
results


Starting async processing...
Processing item: powerbi 2.93.641.0
Processing item: omnii gateway v1.0.0
Processing item: sem 2023.2.1
Processing item: ibm isam aac 10.0.2
Processing item: ibm isam rp 10.0.2
Processing item: telemedicine 1.0.0
Processing item: Pan-OS 11.1
Successfully processed 7 items.
Time taken: 151.65 seconds
Async processing completed.


[{'Name': 'Microsoft Power BI Desktop 2.93.641.0',
  'Summary': 'Microsoft Power BI Desktop follows a rolling release model with monthly updates, meaning that specific older versions like 2.93.641.0 are no longer actively supported once newer versions are released.',
  'Hardware/Software': 'Software',
  'Support Model': 'Rolling',
  'EOS Date': None,
  'Support Tiers': [],
  'Source URLs': ['https://learn.microsoft.com/power-bi/fundamentals/desktop-latest-update',
   'https://chocolatey.org/packages/powerbi-desktop/2.93.641.0'],
  'Confidence': 1.0},
 {'Name': 'omnii gateway v1.0.0',
  'Summary': "No specific End-of-Support (EOS) date could be found for a product explicitly named 'omnii gateway v1.0.0'. The 'Omnii' product line is associated with Zebra Technologies (formerly Motorola Solutions) and primarily consists of handheld computers (hardware).",
  'Hardware/Software': 'Hardware',
  'Support Model': 'Fixed',
  'EOS Date': 'No EOS found',
  'Support Tiers': [],
  'Source URLs': []

In [11]:
def processing_tiers(df):
    df = pd.DataFrame(df)
    expl = df.explode('Support Tiers').reset_index(drop=True)
    tiers_df = pd.json_normalize(expl['Support Tiers'])
    final_df = expl.drop(columns=['Support Tiers']).join(tiers_df)
    return final_df

df_to_csv = processing_tiers(results)
df_to_csv['Source URLs'] = df_to_csv['Source URLs'].str.join(',')

In [70]:
df_to_csv.to_csv('omnii_output1.csv', index=False)

In [12]:
df_to_csv


,Name,Summary,Hardware/Software,Support Model,EOS Date,Source URLs,Confidence
0,Microsoft Power BI Desktop 2.93.641.0,Microsoft Power BI Desktop follows a rolling r...,Software,Rolling,None,https://learn.microsoft.com/power-bi/fundament...,1.0
1,omnii gateway v1.0.0,No specific End-of-Support (EOS) date could be...,Hardware,Fixed,No EOS found,,0.0
2,SolarWinds Security Event Manager 2023.2.1,SolarWinds Security Event Manager version 2023...,Software,Version-Based,No EOS found,https://documentation.solarwinds.com/en/succes...,0.0
3,IBM Security Verify Access Advanced Access Con...,"IBM Security Verify Access, which includes Adv...",Software,Rolling,None,https://vertexaisearch.cloud.google.com/ground...,1.0
4,IBM Security Verify Access 10.0.2 Reverse Proxy,IBM Security Verify Access 10.0.2 ceased recei...,Software,Version-Based,2021-12-17,https://vertexaisearch.cloud.google.com/ground...,1.0
5,telemedicine 1.0.0,No specific End-of-Support (EOS) date could be...,Software,No EOS found,None,,0.0
6,Pan-OS 11.1,"Pan-OS 11.1, a software release for Palo Alto ...",Software,Version-Based,2027-05-03,https://vertexaisearch.cloud.google.com/ground...,1.0


### Plot

In [ ]:
tm = time.time()
time.sleep(1)
tm2 = time.time()- tm
# round to 2 decimal points
 

Time taken: 1.00 seconds


In [137]:
#df to csv

omnii_eos_sw = pd.read_csv('omnii_eos.csv')
omnii_eos_sw.head(5)

,Name,Summary,Hardware/Software,EOS Date,Source URLs,Confidence
0,Product Not Specified,No End-of-Support (EOS) date could be determin...,Software,No EOS found,[],0.0
1,.NET Framework 4.8.1,The .NET Framework 4.8.1 is a software compone...,Software,No specific end date (tied to Windows OS lifec...,['https://learn.microsoft.com/en-us/lifecycle/...,1.0
2,7-Zip 19.00,"7-Zip, as open-source software, does not have ...",Software,No EOS found,"['https://www.7-zip.org/support.html', 'https:...",1.0
3,Google Chrome 115.0.5790.102,Google Chrome 115.0.5790.102 reached its end-o...,Software,2023-08-15,['https://chromereleases.googleblog.com/2023/0...,1.0
4,Splunk Universal Forwarder 9.0.2,"Splunk Universal Forwarder 9.0.2, as part of t...",Software,2027-06-14,['https://vertexaisearch.cloud.google.com/grou...,1.0


In [1]:
import pandas as pd
import numpy as np # Used for representing null values

# Your list of JSON objects from the API/assistant
data = [
    {
      "Name": "Windows Server 2022",
      "Support Model": "Fixed",
      "EOS Date": "2031-10-14",
      "Support Tiers": [
          {"Tier": "Mainstream Support", "EndDate": "2026-10-13"},
          {"Tier": "Extended Support", "EndDate": "2031-10-14"}
      ]
    },
    {
      "Name": "Ubuntu 22.04 LTS",
      "Support Model": "Fixed",
      "EOS Date": "2032-04-30",
      "Support Tiers": [
          {"Tier": "Standard Security", "EndDate": "2027-04-30"},
          {"Tier": "Extended Security Maintenance (ESM)", "EndDate": "2032-04-30"}
      ]
    },
    {
      "Name": "Google Chrome",
      "Support Model": "Rolling",
      "EOS Date": None,
      "Support Tiers": [] # No fixed tiers
    }
]

# 1. Create the initial DataFrame
df = pd.DataFrame(data)

# 2. Explode the 'Support Tiers' column to create separate rows
#    Products with no tiers (like Chrome) will result in a row with NaN
df_exploded = df.explode('Support Tiers').reset_index(drop=True)

# 3. Normalize the 'Support Tiers' column (which now contains dictionaries)
#    and join it back to the main data
tiers_df = pd.json_normalize(df_exploded['Support Tiers'])
final_df = df_exploded.drop(columns=['Support Tiers']).join(tiers_df)

# Optional: Convert date strings to datetime objects for calculations
final_df['EOS Date'] = pd.to_datetime(final_df['EOS Date'])
final_df['EndDate'] = pd.to_datetime(final_df['EndDate'])

print(final_df)

                  Name Support Model   EOS Date  \
0  Windows Server 2022         Fixed 2031-10-14   
1  Windows Server 2022         Fixed 2031-10-14   
2     Ubuntu 22.04 LTS         Fixed 2032-04-30   
3     Ubuntu 22.04 LTS         Fixed 2032-04-30   
4        Google Chrome       Rolling        NaT   

                                  Tier    EndDate  
0                   Mainstream Support 2026-10-13  
1                     Extended Support 2031-10-14  
2                    Standard Security 2027-04-30  
3  Extended Security Maintenance (ESM) 2032-04-30  
4                                  NaN        NaT  


In [2]:

final_df

,Name,Support Model,EOS Date,Tier,EndDate
0,Windows Server 2022,Fixed,2031-10-14,Mainstream Support,2026-10-13
1,Windows Server 2022,Fixed,2031-10-14,Extended Support,2031-10-14
2,Ubuntu 22.04 LTS,Fixed,2032-04-30,Standard Security,2027-04-30
3,Ubuntu 22.04 LTS,Fixed,2032-04-30,Extended Security Maintenance (ESM),2032-04-30
4,Google Chrome,Rolling,NaT,NaN,NaT
